Projekt 1

In [ ]:
# Erster Blick

import pandas as pd

# Erklärung: CSV einlesen (liegt bei dir im gleichen Ordner wie das Notebook)
df = pd.read_csv("Marktkampagne.csv")

# Erklärung: Grundcheck (Zeilen/Spalten) + erste 5 Zeilen anzeigen
print("Shape (Zeilen, Spalten):", df.shape)
df.head()



In [ ]:
# Erklärung: Schneller Überblick über Spalten + Datentypen
df.info()

# Erklärung: Fehlende Werte pro Spalte (absteigend sortiert)
df.isna().sum().sort_values(ascending=False).head(15)


In [ ]:
# Erklärung: Welche Spalten sind Text/Kategorien? (und wie viele Ausprägungen gibt’s?)
df.select_dtypes(include="object").nunique()




In [ ]:
# Erklärung: Datumsformat kurz checken (noch nicht umwandeln, nur anschauen)
df["Datum_Kunde"].head(10)

In [ ]:
# Erklärung: Datum_Kunde ist im Format TT-MM-JJJJ -> dayfirst=True
df["Datum_Kunde"] = pd.to_datetime(df["Datum_Kunde"], dayfirst=True, errors="raise")

# Erklärung: kurzer Sichtcheck
df[["Datum_Kunde"]].head()

In [ ]:
# Erklärung: Stichtag = spätestes Datum im Datensatz (bleibt "dataset-intern", kein 2026-Offset)
stichtag = df["Datum_Kunde"].max()

# Erklärung: Kundendauer in Tagen seit Aufnahme (je größer, desto länger Kunde)
df["Kundendauer_Tage"] = (stichtag - df["Datum_Kunde"]).dt.days

# Erklärung: kurzer Check
df[["Datum_Kunde", "Kundendauer_Tage"]].head()


In [ ]:
# Erklärung: Flag behalten die Info "fehlte ursprünglich" (oft nützlich fürs Modell)
df["Einkommen_fehlte"] = df["Einkommen"].isna().astype(int)

In [ ]:
# Erklärung: fehlendes Einkommen mit Median auffüllen (robust gegen Ausreißer)
df["Einkommen"] = df["Einkommen"].fillna(df["Einkommen"].median())

In [ ]:
# Erklärung: Kontrolle (sollte 0 sein)
df["Einkommen"].isna().sum()

In [ ]:
#  Zelle 1 – Einlesen + Mini-Clean (Datum & Einkommen)

import pandas as pd
import numpy as np


# Erklärung: Datensatz laden
df = pd.read_csv("Marktkampagne.csv")

# Erklärung: Datum sauber machen (falls noch Text)
if df["Datum_Kunde"].dtype == "object":
    df["Datum_Kunde"] = pd.to_datetime(df["Datum_Kunde"], dayfirst=True, errors="raise")

# Erklärung: Einkommen-Fehlwerte (Flag + Median)
if "Einkommen_fehlte" not in df.columns:
    df["Einkommen_fehlte"] = df["Einkommen"].isna().astype(int)

df["Einkommen"] = df["Einkommen"].fillna(df["Einkommen"].median())

df.shape


In [ ]:
# Zelle 2 – Basis-Kennzahlen für Targets (Käufe, Rabatt-Anteil)
# Erklärung: Gesamtanzahl Käufe (über Kanäle)
df["Kaeufe_Gesamt"] = (
    df["Anzahl_Webkäufe"] + df["Anzahl_Katalogkäufe"] + df["Anzahl_Ladeneinkäufe"]
)

# Erklärung: Anteil Rabattkäufe (cap bei 1.0, weil Rabattkäufe teils > Gesamt vorkommen kann)
rabatt_anteil = df["Anzahl_Rabattkäufe"] / df["Kaeufe_Gesamt"].replace(0, np.nan)
df["Rabattanteil"] = rabatt_anteil.clip(upper=1.0).fillna(0.0)

df[["Kaeufe_Gesamt", "Anzahl_Rabattkäufe", "Rabattanteil"]].head()


In [ ]:
# Zelle 3 – Schwellen “datengetrieben” setzen (ohne Raten)

# Erklärung: Churn-Schwelle: wir nehmen Top-10% der höchsten "Letzter_Kauf_Tage" als Risiko
churn_schwelle = df["Letzter_Kauf_Tage"].quantile(0.90)

# Erklärung: Deal-Jäger: Top-20% beim Rabattanteil (recht knackig, aber sinnvoll)
deal_schwelle = df["Rabattanteil"].quantile(0.80)

# Erklärung: Cross-Sell-Proxy: "Fisch-Heavy-Buyer" = Top-20% bei Ausgaben_Fisch
cross_schwelle = df["Ausgaben_Fisch"].quantile(0.80)

churn_schwelle, deal_schwelle, cross_schwelle


In [ ]:
# Zelle 4 – Targets bauen + Verteilungen checken
# (A) Churn-Risiko
df["Ziel_ChurnRisiko"] = (df["Letzter_Kauf_Tage"] >= churn_schwelle).astype(int)

# (B) Deal-Jäger
df["Ziel_DealJaeger"] = (df["Rabattanteil"] >= deal_schwelle).astype(int)

# (C) Cross-Sell-Proxy (Fisch-Propensity = Heavy Buyer)
df["Ziel_Cross_FischTop20"] = (df["Ausgaben_Fisch"] >= cross_schwelle).astype(int)

# Erklärung: Anteil 1er (wie "hart" die Targets sind)
df[["Ziel_ChurnRisiko", "Ziel_DealJaeger", "Ziel_Cross_FischTop20"]].mean()




Churn-Target

In [ ]:
import pandas as pd
import numpy as np

# Erklärung: Daten laden
df = pd.read_csv("Marktkampagne.csv")

# Erklärung: Datum parsen (TT-MM-JJJJ)
df["Datum_Kunde"] = pd.to_datetime(df["Datum_Kunde"], dayfirst=True, errors="raise")

# Erklärung: Einkommen-Fehlwerte (Flag + Median)
df["Einkommen_fehlte"] = df["Einkommen"].isna().astype(int)
df["Einkommen"] = df["Einkommen"].fillna(df["Einkommen"].median())

# Erklärung: Kundendauer als Feature (Stichtag = spätestes Datum im Datensatz)
stichtag = df["Datum_Kunde"].max()
df["Kundendauer_Tage"] = (stichtag - df["Datum_Kunde"]).dt.days

# Erklärung: Churn-Risiko datengetrieben definieren: Top-10% der höchsten Letzter_Kauf_Tage
churn_schwelle = df["Letzter_Kauf_Tage"].quantile(0.90)
df["Ziel_ChurnRisiko"] = (df["Letzter_Kauf_Tage"] >= churn_schwelle).astype(int)

print("Churn-Schwelle (Tage seit letztem Kauf):", churn_schwelle)
print("Churn-Rate (Anteil Ziel=1):", df["Ziel_ChurnRisiko"].mean())
df[["Letzter_Kauf_Tage", "Ziel_ChurnRisiko"]].head()


Feature-Set für Churn

In [ ]:
# Wichtig: Weil das Target aus Letzter_Kauf_Tage definiert wird, darf diese Spalte nicht als Feature rein (sonst “schummelt” das Modell).

In [ ]:
# Erklärung: Features = alles außer ID, Datum und alles was direkt das Target verrät
ziel = "Ziel_ChurnRisiko"

leakage_cols = ["ID", "Datum_Kunde", "Letzter_Kauf_Tage", ziel]
X = df.drop(columns=leakage_cols)
y = df[ziel]

# Erklärung: Spalten-Typen bestimmen
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

cat_cols, len(num_cols)


Baseline-Modell (Logistische Regression) + Mini-Auswertung

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# Erklärung: Train/Test Split (stratify hält die Churn-Rate stabil)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Erklärung: Preprocessing (Kategorien OneHot, Zahlen skalieren)
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

# Erklärung: Modell-Pipeline
model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

# Erklärung: Bewertung (ROC-AUC ist bei unbalancierten Klassen oft sinnvoll)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print("\nReport:\n", classification_report(y_test, pred))


Opti für Churn

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

ziel = "Ziel_ChurnRisiko"

# Erklärung: Falls Ziel noch nicht existiert -> bauen (Top-10% Recency als Risiko)
if ziel not in df.columns:
    churn_schwelle = df["Letzter_Kauf_Tage"].quantile(0.90)
    df[ziel] = (df["Letzter_Kauf_Tage"] >= churn_schwelle).astype(int)

# Erklärung: Schalter: True = praxisnah (Recency drin), False = strikt (Recency raus)
RECENCY_ALS_FEATURE = True

drop_cols = ["ID", "Datum_Kunde", ziel]
if not RECENCY_ALS_FEATURE:
    drop_cols += ["Letzter_Kauf_Tage"]

X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df[ziel]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Churn-Rate:", y.mean(), "| Train:", y_train.mean(), "| Test:", y_test.mean())
print("Features:", X.shape[1], "| Num:", len(num_cols), "| Cat:", len(cat_cols))


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score, average_precision_score

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, proba))
print("PR-AUC (Average Precision):", average_precision_score(y_test, proba))


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score

thresholds = np.linspace(0.05, 0.95, 19)

rows = []
for t in thresholds:
    pred_t = (proba >= t).astype(int)
    rows.append({
        "threshold": round(float(t), 2),
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t, zero_division=0),
        "f1": f1_score(y_test, pred_t, zero_division=0),
        "f2": fbeta_score(y_test, pred_t, beta=2, zero_division=0),
        "positives_pred": int(pred_t.sum())
    })

res = pd.DataFrame(rows).sort_values("f2", ascending=False)
res.head(10)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# res ist dein DataFrame aus der Threshold-Tabelle
best_row = res[res["recall"] == 1.0].sort_values("threshold", ascending=False).iloc[0]
best_t = float(best_row["threshold"])

pred_best = (proba >= best_t).astype(int)

print("Gewählte Schwelle (max bei Recall=1.0):", best_t)
print(best_row[["precision","recall","f1","f2","positives_pred"]].to_dict())
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_best))
print("\nReport:\n", classification_report(y_test, pred_best))


In [ ]:
# Erklärung: Churn-Risiko für ALLE Kunden berechnen
proba_all = model.predict_proba(X)[:, 1]
df["Churn_Score"] = proba_all
df["Churn_Flag"] = (df["Churn_Score"] >= 0.7).astype(int)

# Erklärung: Top-Risiko-Liste
churn_liste = df.loc[df["Churn_Flag"] == 1, ["ID", "Churn_Score", "Letzter_Kauf_Tage", "Kundendauer_Tage"]].copy()
churn_liste = churn_liste.sort_values("Churn_Score", ascending=False)

churn_liste.head(20)


In [ ]:
import pandas as pd

prep = model.named_steps["prep"]
clf  = model.named_steps["clf"]

feat_names = prep.get_feature_names_out()
coefs = clf.coef_[0]

imp = (pd.DataFrame({"feature": feat_names, "coef": coefs})
       .assign(abs=lambda d: d["coef"].abs())
       .sort_values("abs", ascending=False))

print("Top positiv (macht Churn wahrscheinlicher):")
display(imp.sort_values("coef", ascending=False).head(15)[["feature","coef"]])

print("\nTop negativ (macht Churn unwahrscheinlicher):")
display(imp.sort_values("coef", ascending=True).head(15)[["feature","coef"]])


Opti 2 Churn

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
import numpy as np
import pandas as pd

ziel = "Ziel_ChurnRisiko"

drop_cols = ["ID", "Datum_Kunde", ziel, "Letzter_Kauf_Tage"]  # <- wichtig
X2 = df.drop(columns=[c for c in drop_cols if c in df.columns])
y2 = df[ziel]

cat_cols2 = X2.select_dtypes(include="object").columns.tolist()
num_cols2 = [c for c in X2.columns if c not in cat_cols2]

X_train, X_test, y_train, y_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

preprocess2 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols2),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols2),
    ],
    remainder="drop"
)

model2 = Pipeline(steps=[
    ("prep", preprocess2),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

model2.fit(X_train, y_train)
proba2 = model2.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, proba2))
print("PR-AUC:", average_precision_score(y_test, proba2))


In [ ]:
from sklearn.metrics import precision_score, recall_score, fbeta_score

thresholds = np.linspace(0.05, 0.95, 19)
rows = []
for t in thresholds:
    pred = (proba2 >= t).astype(int)
    rows.append({
        "threshold": round(float(t), 2),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f2": fbeta_score(y_test, pred, beta=2, zero_division=0),
        "positives_pred": int(pred.sum())
    })

res2 = pd.DataFrame(rows).sort_values("f2", ascending=False)
res2.head(10)


In [ ]:
best_t = float(res2.iloc[0]["threshold"])
pred_best = (proba2 >= best_t).astype(int)

print("Beste Schwelle (F2):", best_t)
print(res2.iloc[0][["precision","recall","f2","positives_pred"]].to_dict())
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_best))
print("\nReport:\n", classification_report(y_test, pred_best))


In [ ]:
# Erklärung: Sicherstellen, dass Letzter_Kauf_Tage NICHT in den Features steckt
print("Letzter_Kauf_Tage in X2?", "Letzter_Kauf_Tage" in X2.columns)

# Erklärung: Sicherheitsliste: alles was nach Recency/Letzter Kauf aussieht
[c for c in X2.columns if "letzter" in c.lower() or "recency" in c.lower()]


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = cross_val_score(model2, X2, y2, cv=cv, scoring="roc_auc")
pr_scores  = cross_val_score(model2, X2, y2, cv=cv, scoring="average_precision")

print("ROC-AUC folds:", auc_scores)
print("ROC-AUC mean/std:", auc_scores.mean(), auc_scores.std())

print("\nPR-AUC folds:", pr_scores)
print("PR-AUC mean/std:", pr_scores.mean(), pr_scores.std())


In [ ]:

# Fertige Maschine churn 


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, fbeta_score,
    confusion_matrix, classification_report
)

# =========================
# 1) Daten laden + Mini-Clean
# =========================
# Erklärung: CSV einlesen
df = pd.read_csv("Marktkampagne.csv")

# Erklärung: Datum parsen (TT-MM-JJJJ)
df["Datum_Kunde"] = pd.to_datetime(df["Datum_Kunde"], dayfirst=True, errors="raise")

# Erklärung: Einkommen-Fehlwerte (Flag + Median)
df["Einkommen_fehlte"] = df["Einkommen"].isna().astype(int)
df["Einkommen"] = df["Einkommen"].fillna(df["Einkommen"].median())

# Erklärung: Kundendauer als Feature
stichtag = df["Datum_Kunde"].max()
df["Kundendauer_Tage"] = (stichtag - df["Datum_Kunde"]).dt.days

# =========================
# 2) Churn-Target bauen
# =========================
# Erklärung: Churn-Risiko = Top-10% der höchsten "Letzter_Kauf_Tage"
churn_thr = df["Letzter_Kauf_Tage"].quantile(0.90)
df["Ziel_ChurnRisiko"] = (df["Letzter_Kauf_Tage"] >= churn_thr).astype(int)

print("Churn-Schwelle (Letzter_Kauf_Tage):", float(churn_thr))
print("Churn-Rate (Anteil 1):", df["Ziel_ChurnRisiko"].mean())

# =========================
# 3) Features bauen (Leakage vermeiden!)
# =========================
# Erklärung: Letzter_Kauf_Tage raus, weil daraus das Target definiert wurde
ziel = "Ziel_ChurnRisiko"
drop_cols = ["ID", "Datum_Kunde", ziel, "Letzter_Kauf_Tage"]

X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df[ziel]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# Erklärung: Train/Test Split (stratify hält Klassenverteilung stabil)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train-Rate:", y_train.mean(), "| Test-Rate:", y_test.mean())

# =========================
# 4) Modell (Baseline) + Scores
# =========================
# Erklärung: Preprocessing für Zahlen/Kategorien
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

# Erklärung: Logistic Regression + class_weight=balanced (gegen "immer 0" Problem)
model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]

print("\nROC-AUC:", roc_auc_score(y_test, proba))
print("PR-AUC:", average_precision_score(y_test, proba))

# =========================
# 5) Threshold-Optimierung (F2 -> Recall wichtiger)
# =========================
# Erklärung: Wir suchen die Schwelle, die F2 maximiert (Churn: lieber weniger verpassen)
thresholds = np.linspace(0.05, 0.95, 19)

best = {"t": None, "precision": None, "recall": None, "f2": -1, "positives_pred": None}
for t in thresholds:
    pred = (proba >= t).astype(int)
    p = precision_score(y_test, pred, zero_division=0)
    r = recall_score(y_test, pred, zero_division=0)
    f2 = fbeta_score(y_test, pred, beta=2, zero_division=0)

    if f2 > best["f2"]:
        best = {"t": float(t), "precision": p, "recall": r, "f2": f2, "positives_pred": int(pred.sum())}

pred_best = (proba >= best["t"]).astype(int)

print("\nBeste Schwelle (F2):", best["t"])
print({"precision": best["precision"], "recall": best["recall"], "f2": best["f2"], "positives_pred": best["positives_pred"]})
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_best))
print("\nReport:\n", classification_report(y_test, pred_best))


Churn-Schwelle (Letzter_Kauf_Tage): 89.0
Churn-Rate (Anteil 1): 0.10625
Train-Rate: 0.10602678571428571 | Test-Rate: 0.10714285714285714

ROC-AUC: 0.5015625
PR-AUC: 0.10623558587044257

Beste Schwelle (F2): 0.2
{'precision': 0.11483253588516747, 'recall': 1.0, 'f2': 0.39344262295081966, 'positives_pred': 418}
Confusion Matrix:
 [[ 30 370]
 [  0  48]]

Report:
               precision    recall  f1-score   support

           0       1.00      0.07      0.14       400
           1       0.11      1.00      0.21        48

    accuracy                           0.17       448
   macro avg       0.56      0.54      0.17       448
weighted avg       0.91      0.17      0.15       448



In [ ]:
# Erklärung: Unser "Churn" ist direkt über Recency definiert (Letzter_Kauf_Tage).
# Wenn wir Letzter_Kauf_Tage als Feature entfernen (Leakage vermeiden), bleibt kaum Signal übrig:
# -> ROC-AUC ~ 0.5 / PR-AUC ~ Grundrate => Modell rät praktisch.
# Wenn wir Letzter_Kauf_Tage drin lassen, ist das Modell "zu gut", weil es genau die Ziel-Info enthält:
# -> ML wäre nur eine komplizierte Version einer einfachen Regel.
# Fazit: Für dieses Label ist eine Recency-Regel transparenter, stabiler und sinnvoller als ML.


Deal-Jäger Target bauen (datengetrieben)

In [ ]:
import numpy as np

# Erklärung: Gesamt-Käufe über Kanäle
df["Kaeufe_Gesamt"] = (
    df["Anzahl_Webkäufe"] + df["Anzahl_Katalogkäufe"] + df["Anzahl_Ladeneinkäufe"]
)

# Erklärung: Rabattanteil (Deals / Gesamt), robust gegen 0-Käufe
df["Rabattanteil"] = (
    df["Anzahl_Rabattkäufe"] / df["Kaeufe_Gesamt"].replace(0, np.nan)
).clip(upper=1.0).fillna(0.0)

# Erklärung: Deal-Jäger = Top-20% Rabattanteil
deal_schwelle = df["Rabattanteil"].quantile(0.80)
df["Ziel_DealJaeger"] = (df["Rabattanteil"] >= deal_schwelle).astype(int)

print("Deal-Schwelle (Rabattanteil):", deal_schwelle)
print("Deal-Rate (Anteil Ziel=1):", df["Ziel_DealJaeger"].mean())
df[["Kaeufe_Gesamt", "Anzahl_Rabattkäufe", "Rabattanteil", "Ziel_DealJaeger"]].head()


Features für Deal-Jäger (Leakage raus!)

In [ ]:
ziel = "Ziel_DealJaeger"

leakage_cols = ["ID", "Datum_Kunde", ziel, "Anzahl_Rabattkäufe", "Rabattanteil", "Kaeufe_Gesamt"]
X = df.drop(columns=[c for c in leakage_cols if c in df.columns])
y = df[ziel]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Features:", X.shape[1], "| Kategorien:", len(cat_cols), "| Numerisch:", len(num_cols))


Baseline-Modell + Bewertung

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print("\nReport:\n", classification_report(y_test, pred))


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, confusion_matrix, classification_report

# Erklärung: proba = Modellwahrscheinlichkeiten für Klasse 1 (Deal-Jäger) im Testset
# y_test = echte Labels im Testset
thresholds = np.linspace(0.05, 0.95, 19)

rows = []
for t in thresholds:
    pred = (proba >= t).astype(int)
    rows.append({
        "threshold": round(float(t), 2),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "f0_5": fbeta_score(y_test, pred, beta=0.5, zero_division=0),
        "positives_pred": int(pred.sum())
    })

res = pd.DataFrame(rows).sort_values("f0_5", ascending=False)
res.head(10)


In [ ]:
best = res.iloc[0]
best_t = float(best["threshold"])

pred_best = (proba >= best_t).astype(int)

print("Beste Schwelle (F0.5):", best_t)
print(best[["precision","recall","f1","f0_5","positives_pred"]].to_dict())
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_best))
print("\nReport:\n", classification_report(y_test, pred_best))


In [ ]:
import pandas as pd

prep = model_deal.named_steps["prep"]
clf  = model_deal.named_steps["clf"]

feat_names = prep.get_feature_names_out()
coefs = clf.coef_[0]

imp = (pd.DataFrame({"feature": feat_names, "coef": coefs})
       .assign(abs=lambda d: d["coef"].abs())
       .sort_values("abs", ascending=False))

print("Top positiv (macht Deal-Jäger wahrscheinlicher):")
display(imp.sort_values("coef", ascending=False).head(15)[["feature","coef"]])

print("\nTop negativ (macht Deal-Jäger unwahrscheinlicher):")
display(imp.sort_values("coef", ascending=True).head(15)[["feature","coef"]])


Cross-Target bauen (Kategorie automatisch finden)

In [ ]:
# Erklärung: Eine passende "Ausgaben_*" oder "Mnt*Products" Spalte automatisch finden
prioritaet = [
    "Ausgaben_Fisch", "Ausgaben_Wein", "Ausgaben_Fleisch", "Ausgaben_Gold", "Ausgaben_Süßes",
    "MntFishProducts", "MntWines", "MntMeatProducts", "MntGoldProds", "MntSweetProducts"
]

spend_cols = [c for c in df.columns if c.lower().startswith("ausgaben_")]
if not spend_cols:
    spend_cols = [c for c in df.columns if c.startswith("Mnt")]

cross_col = next((c for c in prioritaet if c in df.columns), None)
if cross_col is None:
    cross_col = spend_cols[0]  # nimmt die erste passende Ausgaben/Mnt-Spalte

print("Cross-Kategorie-Spalte:", cross_col)

# Erklärung: Target = Top-20% in dieser Kategorie (robust, falls viele 0-Werte)
nonzero = df[cross_col] > 0
if nonzero.mean() >= 0.20:
    cross_schwelle = df[cross_col].quantile(0.80)
    df["Ziel_Cross_Top20"] = (df[cross_col] >= cross_schwelle).astype(int)
else:
    # wenn fast alle 0 sind: Top-20% innerhalb der Nicht-Nullen
    cross_schwelle = df.loc[nonzero, cross_col].quantile(0.80)
    df["Ziel_Cross_Top20"] = ((df[cross_col] >= cross_schwelle) & nonzero).astype(int)

print("Cross-Schwelle:", cross_schwelle)
print("Cross-Rate (Anteil Ziel=1):", df["Ziel_Cross_Top20"].mean())


Features vorbereiten (Leakage raus)

In [ ]:
# Erklärung: Für Cross darf die Ziel-Kategorie-Spalte NICHT als Feature rein
ziel = "Ziel_Cross_Top20"

leakage_cols = ["ID", "Datum_Kunde", ziel, cross_col, "Ausgaben_Gesamt", "MntTotal"]  # optional, nur falls vorhanden
X = df.drop(columns=[c for c in leakage_cols if c in df.columns])
y = df[ziel]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Features:", X.shape[1], "| Kategorien:", len(cat_cols), "| Numerisch:", len(num_cols))


Baseline-Modell (LogReg) + Bewertung

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# Erklärung: Split mit stratify, damit Klassenverteilung gleich bleibt
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print("\nReport:\n", classification_report(y_test, pred))


Cross Optimiertung

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.linspace(0.05, 0.95, 91)

rows = []
for t in thresholds:
    pred_t = (proba >= t).astype(int)
    rows.append({
        "threshold": round(float(t), 2),
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t, zero_division=0),
        "f1": f1_score(y_test, pred_t, zero_division=0),
        "positives_pred": int(pred_t.sum())
    })

res = pd.DataFrame(rows)

# Erklärung: Kandidaten mit Precision >= 0.80, davon den mit höchstem Recall nehmen
ziel_precision = 0.80
cand = res[res["precision"] >= ziel_precision].sort_values(["recall", "f1"], ascending=False)

cand.head(10)


Beste gefundene Schwelle anwenden + neue Matrix/Report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

if len(cand) == 0:
    print("Keine Schwelle erreicht Precision >=", ziel_precision, "— nimm z.B. 0.75 oder senke Zielprecision.")
else:
    best_t = float(cand.iloc[0]["threshold"])
    pred_best = (proba >= best_t).astype(int)

    print("Gewählte Schwelle:", best_t)
    print("Precision/Recall/F1:", cand.iloc[0][["precision","recall","f1"]].to_dict())
    print("Positives predicted:", int(cand.iloc[0]["positives_pred"]))
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred_best))
    print("\nReport:\n", classification_report(y_test, pred_best))
